In [ ]:
import os
import numpy as np
import cytoflow as flow

# Configuration
OUTPUT_DIR = "outputfolder"
PARAMS = sorted(["CD10", "CD19", "CD20", "CD34", "CD38", "CD45", "CD58", "CD66"])


def choose_parameters(lista, chosen, channels):
    """
    Select the required parameters from the metadata.
    Returns None if any are missing.
    """
    samplepar = []
    columnname = []

    for param in chosen:
        selected = None 

        for markers in lista:
            name = markers[0]

            if param == name:
                if name in channels:
                    selected = name
                else:
                    selected = markers[-1]
                break

        if selected is None:
            return None, None  

        samplepar.append(selected)
        columnname.append(param)

    return samplepar, columnname


def read_fcs_file(datafile):
    tube = flow.Tube(file=datafile)

    exp = None
    for metadata_key in ["$PnN", "$PnS"]:
        try:
            import_op = flow.ImportOp(tubes=[tube], name_metadata=metadata_key)
            exp = import_op.apply()
            break
        except KeyError:
            continue

    if exp is None:
        print(f"ERROR: metadata not readable in {datafile}")
        return None

    metadata = exp.metadata["fcs_metadata"][datafile]
    npar = metadata["$PAR"]

    # ---
    lista = []
    for i in range(1, npar + 1):
        key_s = f"$P{i}S"
        key_n = f"$P{i}N"

        if key_s in metadata:
            aux = metadata[key_s]

            if aux.isdecimal() or (aux and aux[0].isdigit() and aux.endswith(("A", "B", "C"))):
                lista.append([f"CD{aux}", metadata[key_n]])
            else:
                lista.append([aux, metadata[key_n]])
        else:
            lista.append([metadata[key_n]])

    # ---
    samplepar, colnames = choose_parameters(lista, PARAMS, exp.channels)

    if samplepar is None:
        print(f"Missing markers in {datafile}")
        return None

    # ---
    try:
        df = exp.data[samplepar]
        df.columns = colnames
    except Exception as e:
        print(f"Error extracting data from {datafile}: {e}")
        return None

    return df


def process_directory(input_dir):
    """
    Searches patient directories and converts FCS to TXT.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for patient in sorted(os.listdir(input_dir)):
        if patient.startswith("."):
            continue

        patient_in = os.path.join(input_dir, patient)
        patient_out = os.path.join(OUTPUT_DIR, patient)

        os.makedirs(patient_out, exist_ok=True)

        for idx, filename in enumerate(sorted(os.listdir(patient_in))):
            if filename.startswith("."):
                continue

            filepath = os.path.join(patient_in, filename)

            df = read_fcs_file(filepath)

            if df is None:
                print(f"Skipped: {filename}")
                continue

            tube_name = filename.split(".")[0]
            tube_dir = os.path.join(patient_out, tube_name)

            os.makedirs(tube_dir, exist_ok=True)

            output_file = os.path.join(tube_dir, f"Tube{idx}.txt")

            np.savetxt(output_file, df.values, fmt="%f")
    
    print("Ready")


# MAIN


if __name__ == "__main__":
    input_dir = "inputfolder"  
    process_directory(input_dir)


Ready
